In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"D:\Machine Learning\Datasets\Cases\Glass_Identification")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

# sklearn imports
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# Load data
glass = pd.read_csv('Glass.csv')

# Features: all columns except 'Type'
X = glass.drop('Type', axis=1)

# Target: encode Type as integer labels (0 to 5)
le = LabelEncoder()
y = le.fit_transform(glass['Type'])

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=25, stratify=y
)

# Scale features
scaler = MinMaxScaler()
X_scl_trn = scaler.fit_transform(X_train)
X_scl_tst = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_scl_trn).float()
y_train_tensor = torch.from_numpy(y_train).long()        # IMPORTANT: long for class indices
X_test_tensor = torch.from_numpy(X_scl_tst).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Smaller batch size

# Model definition
# torch.manual_seed(25)
# model = nn.Sequential(
#     nn.Linear(in_features=X_scl_trn.shape[1], out_features=18),
#     nn.ReLU(),
#     nn.Linear(in_features=18, out_features=6)  # 6 classes
# )
class MLPClassifier(torch.nn.Module):
    def __init__(self,num_features):
        super(MLPClassifier, self).__init__()
        self.linear1 = nn.Linear(in_features=num_features, out_features=18)
        self.linear2 = nn.Linear(in_features=18, out_features=7)
        # self.linear3 = nn.Linear(in_features=7, out_features=3)
        self.linear3 = nn.Linear(in_features=7, out_features=6)
        self.act1 = nn.ReLU()
    
    def forward(self,x):
        x= self.linear1(x)
        x= self.act1(x)
        x= self.linear2(x)
        x= self.act1(x)
        output= self.linear3(x)
        # x= self.act1(x)
        # output= self.linear4(x)
        return output
# Model summary (correct input shape)
model = MLPClassifier(num_features = X_scl_trn.shape[1])

# Print model summary
summary(model, input_size=(X_scl_trn.shape[1],))  # Correct shape

# Loss and optimizer
criterion = nn.CrossEntropyLoss()          # Expects raw logits and long labels
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# Training loop
num_epochs = 1000
for epoch in range(num_epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)           # Raw logits
        
        # Compute loss
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
    
    # Print loss every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluation on test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    
    # Probabilities
    softmax = nn.Softmax(dim=1)
    test_proba = softmax(test_outputs)
    
    # Predicted class
    y_pred = torch.argmax(test_outputs, dim=1).numpy()  # Directly from logits is fine
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f'\nTest Accuracy: {accuracy:.4f}')
    
    # Log loss (using probabilities)
    test_proba_np = test_proba.numpy()
    test_log_loss = log_loss(y_test, test_proba_np)
    print(f'Test Log Loss: {test_log_loss:.4f}')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 18]             180
              ReLU-2                   [-1, 18]               0
            Linear-3                    [-1, 7]             133
              ReLU-4                    [-1, 7]               0
            Linear-5                    [-1, 6]              48
Total params: 361
Trainable params: 361
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
Epoch [100/1000], Loss: 1.3878
Epoch [200/1000], Loss: 1.1584
Epoch [300/1000], Loss: 0.8364
Epoch [400/1000], Loss: 0.6990
Epoch [500/1000], Loss: 0.9634
Epoch [600/1000], Loss: 0.6303
Epoch [700/1000], Loss: 0.7917
Epoch [800/1000], Loss: 0